# BirdCLEF+ 2026 — SED Training

EfficientNet-B0 + AttentionSEDHead で学習する自己完結型ノートブック。
aidensong123 の SED Baseline (LB 0.862) のアーキテクチャを再現。

## 改善点
- **Silero-VAD** で人の声を検出→除去（前処理）
- **num_classes=234**（ゼロショット種もヘッドに含む）

## Kaggle Settings
| Setting | Value |
|---------|-------|
| Data | `birdclef-2026` (Competition) |
| Accelerator | GPU T4 x2 |
| Internet | ON |

In [ ]:
import os, time, glob, random, pickle, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import timm
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from dataclasses import dataclass

warnings.filterwarnings('ignore')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# CONFIG
# ============================================================
COMP_DATA = None
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p): COMP_DATA = p; break
assert COMP_DATA

@dataclass
class Config:
    # Audio
    sr: int = 32000              # Hz: サンプリング周波数
    chunk_duration: float = 5.0  # 秒: 入力窓長
    n_mels: int = 256            # メルバンド数
    n_fft: int = 2048            # FFT窓サイズ
    hop_length: int = 512        # FFTホップ長
    fmin: int = 0                # Hz: メルフィルタ最低周波数
    fmax: int = 16000            # Hz: メルフィルタ最高周波数
    top_db: float = 80.0         # dB: AmplitudeToDB の上限
    power: float = 2.0           # パワースペクトログラム
    target_size: tuple = (256, 256)  # リサイズ後のスペクトログラムサイズ
    # Model
    backbone: str = 'tf_efficientnet_b0.ns_jft_in1k'  # Noisy Student 事前学習済み
    num_classes: int = 234       # 提出要求の全234種
    in_channels: int = 3         # RGB 3ch（ImageNet 事前学習に合わせる）
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0      # GeM pooling の初期値
    # Training
    num_epochs: int = 10
    batch_size: int = 32
    lr: float = 1e-3
    lr_min: float = 1e-6
    weight_decay: float = 1e-4
    seed: int = 42
    fp16: bool = True
    n_folds: int = 5
    early_stopping: int = 3
    # Augmentation
    mixup_alpha: float = 0.4
    use_silero_vad: bool = True  # 人の声除去
    # Paths
    train_audio: str = f'{COMP_DATA}/train_audio'
    train_csv: str = f'{COMP_DATA}/train.csv'
    output_dir: str = '/kaggle/working'

    @property
    def chunk_samples(self): return int(self.sr * self.chunk_duration)

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(cfg.seed)
print(f'Device: {device}')
print(f'Config: epochs={cfg.num_epochs}, batch={cfg.batch_size}, backbone={cfg.backbone}')

In [ ]:
# ============================================================
# MODEL: EfficientNet-B0 + GeM freq pool + Attention SED Head
# ============================================================
class GEMFreqPool(nn.Module):
    """周波数方向の Generalized Mean Pooling。(B,C,F,T)→(B,C,T)"""
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps
    def forward(self, x):
        p = self.p.clamp(min=1.0)
        return x.clamp(min=self.eps).pow(p).mean(dim=2).pow(1.0 / p)

class AttentionSEDHead(nn.Module):
    """時間方向の Attention Pooling + 分類。
    各時間フレームの重要度(attention)を計算し、重み付き合計で clipwise 予測を出す。"""
    def __init__(self, feat_dim, num_classes, dropout=0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim), nn.ReLU(inplace=True), nn.Dropout(dropout))
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)  # 時間重要度
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)  # フレーム分類
    def forward(self, x):
        x = self.fc(x.permute(0, 2, 1)).permute(0, 2, 1)
        att = F.softmax(torch.tanh(self.att_conv(x)), dim=-1)  # (B, C, T)
        cls = self.cls_conv(x)                                  # (B, C, T)
        clipwise = (att * cls).sum(dim=-1)                      # (B, C)
        return {'clipwise_logit': clipwise, 'framewise_logit': cls.permute(0, 2, 1)}

class SEDModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone, pretrained=True, in_chans=cfg.in_channels,
            features_only=False, global_pool='', num_classes=0,
            drop_path_rate=cfg.drop_path_rate)
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(self.backbone.num_features, cfg.num_classes, cfg.dropout)
    def forward(self, x):
        return self.head(self.gem_pool(self.backbone(x)))

print('Model definition OK')

In [ ]:
# ============================================================
# MEL TRANSFORM + SILERO-VAD
# ============================================================
class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
            n_mels=cfg.n_mels, f_min=cfg.fmin, f_max=cfg.fmax,
            power=cfg.power)
        self.db = T.AmplitudeToDB(stype='power', top_db=cfg.top_db)
        self.resize = torchvision.transforms.Resize(cfg.target_size, antialias=True)
    @torch.no_grad()
    def forward(self, waveforms):
        mel = self.db(self.mel(waveforms))
        mel = self.resize(mel)
        B = mel.shape[0]
        mel_flat = mel.reshape(B, -1)
        mn = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mx = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mn) / (mx - mn + 1e-7)  # [0,1] 正規化
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)  # (B,3,H,W)

mel_transform = MelSpectrogramTransform(cfg)

# Silero-VAD: 人の声区間を検出
vad_model = None
get_speech_ts = None
if cfg.use_silero_vad:
    try:
        vad_model, utils = torch.hub.load('snakers4/silero-vad', 'silero_vad')
        get_speech_ts = utils[0]  # get_speech_timestamps
        print('Silero-VAD loaded OK')
    except Exception as e:
        print(f'Silero-VAD failed: {e}')
        cfg.use_silero_vad = False

def remove_speech(waveform, sr=32000):
    """人の声区間をゼロに置換する。"""
    if vad_model is None: return waveform
    # Silero-VAD は 16kHz を期待
    wf_16k = librosa.resample(waveform, orig_sr=sr, target_sr=16000)
    wf_tensor = torch.tensor(wf_16k, dtype=torch.float32)
    try:
        timestamps = get_speech_ts(wf_tensor, vad_model, sampling_rate=16000)
        for ts in timestamps:
            # 16kHz → 32kHz に変換
            start = int(ts['start'] * sr / 16000)
            end = int(ts['end'] * sr / 16000)
            waveform[start:end] = 0.0
    except Exception:
        pass
    return waveform

print('Mel transform + VAD OK')

In [ ]:
# ============================================================
# DATASET
# ============================================================
class BirdSEDDataset(Dataset):
    def __init__(self, df, cfg, le, mode='train'):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.le = le
        self.mode = mode
        self.num_classes = cfg.num_classes
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.cfg.train_audio, row['filename'])
        try:
            waveform, _ = librosa.load(path, sr=self.cfg.sr, mono=True)
        except:
            waveform = np.zeros(self.cfg.chunk_samples, dtype=np.float32)
        # Silero-VAD: 人の声除去
        if self.mode == 'train' and self.cfg.use_silero_vad:
            waveform = remove_speech(waveform, self.cfg.sr)
        # 5秒にクロップ/パディング
        target = self.cfg.chunk_samples
        if len(waveform) > target:
            if self.mode == 'train':
                start = random.randint(0, len(waveform) - target)
            else:
                start = 0
            waveform = waveform[start:start + target]
        elif len(waveform) < target:
            waveform = np.pad(waveform, (0, target - len(waveform)))
        # Label (multi-hot)
        label = np.zeros(self.num_classes, dtype=np.float32)
        try:
            label[self.le.transform([str(row['primary_label'])])[0]] = 1.0
        except ValueError:
            pass
        return torch.tensor(waveform, dtype=torch.float32), torch.tensor(label)

print('Dataset OK')

In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================
def train_one_epoch(model, loader, criterion, optimizer, scaler, mel_tf, device, cfg, epoch):
    model.train()
    mel_tf = mel_tf.to(device)
    total_loss, n = 0, len(loader)
    t0 = time.time()
    for i, (waveforms, labels) in enumerate(loader):
        waveforms = waveforms.to(device)
        labels = labels.to(device)
        mel = mel_tf(waveforms)
        optimizer.zero_grad(set_to_none=True)
        if cfg.fp16:
            with autocast():
                out = model(mel)
                loss = criterion(out['clipwise_logit'], labels)
                # framewise loss: 各フレームの最大値にも BCE をかける
                fw_max = out['framewise_logit'].max(dim=1)[0]
                loss = loss + criterion(fw_max, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            out = model(mel)
            loss = criterion(out['clipwise_logit'], labels)
            fw_max = out['framewise_logit'].max(dim=1)[0]
            loss = loss + criterion(fw_max, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item()
        if (i+1) % 100 == 0 or (i+1) == n:
            elapsed = time.time() - t0
            print(f'  Epoch {epoch:02d} [{i+1}/{n}] loss={loss.item():.4f} '
                  f'elapsed={elapsed/60:.1f}m', flush=True)
    return total_loss / n

@torch.no_grad()
def validate(model, loader, criterion, mel_tf, device, cfg):
    model.eval()
    mel_tf = mel_tf.to(device)
    total_loss = 0
    all_preds, all_labels = [], []
    for waveforms, labels in loader:
        waveforms = waveforms.to(device)
        labels = labels.to(device)
        mel = mel_tf(waveforms)
        if cfg.fp16:
            with autocast():
                out = model(mel)
                loss = criterion(out['clipwise_logit'], labels)
        else:
            out = model(mel)
            loss = criterion(out['clipwise_logit'], labels)
        total_loss += loss.item()
        all_preds.append(torch.sigmoid(out['clipwise_logit']).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    y_true = np.vstack(all_labels)
    y_pred = np.vstack(all_preds)
    # Padded cmAP
    aps = []
    for c in range(y_true.shape[1]):
        idx = np.argsort(y_pred[:, c])[::-1]
        ts = y_true[:, c][idx]
        padded = np.concatenate([np.ones(5, dtype=np.float32), ts])
        tp_cum = np.cumsum(padded)
        prec = tp_cum / (np.arange(len(padded)) + 1)
        aps.append(np.sum(prec * padded) / (tp_cum[-1] + 1e-10))
    return total_loss / len(loader), float(np.mean(aps))

print('Training loop OK')

In [ ]:
# ============================================================
# RUN TRAINING — Fold 0
# ============================================================
df = pd.read_csv(cfg.train_csv).dropna(subset=['primary_label']).reset_index(drop=True)
print(f'Samples: {len(df)}')

# LabelEncoder: 提出要求234種全てを含む
sub = pd.read_csv(f'{COMP_DATA}/sample_submission.csv')
all_species = sorted([c for c in sub.columns if c != 'row_id'])
le = LabelEncoder()
le.fit(all_species)
print(f'Classes: {len(le.classes_)}')

# Fold 0
skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
train_idx, val_idx = list(skf.split(df, df['primary_label']))[0]
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

train_ds = BirdSEDDataset(train_df, cfg, le, 'train')
val_ds = BirdSEDDataset(val_df, cfg, le, 'val')
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=0, pin_memory=True)
print(f'Batches: train={len(train_loader)}, val={len(val_loader)}')

model = SEDModel(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.num_epochs, eta_min=cfg.lr_min)
scaler = GradScaler(enabled=cfg.fp16)
criterion = nn.BCEWithLogitsLoss()

best_score, patience = -1.0, 0
for epoch in range(1, cfg.num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler,
                                 mel_transform, device, cfg, epoch)
    val_loss, val_cmap = validate(model, val_loader, criterion, mel_transform, device, cfg)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | '
          f'val_cmap={val_cmap:.4f} | lr={lr:.2e}', flush=True)
    if val_cmap > best_score:
        best_score, patience = val_cmap, 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_score': val_cmap, 'metrics': {'cmap': val_cmap}},
                   f'{cfg.output_dir}/best_fold0.pt')
        print(f'  -> New best: {best_score:.4f}', flush=True)
    else:
        patience += 1
        if patience >= cfg.early_stopping:
            print(f'  Early stopping at epoch {epoch}'); break

# Save label encoder
with open(f'{cfg.output_dir}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print(f'\nFold 0 Best cmAP: {best_score:.4f}')

In [ ]:
# Verify outputs
for f in sorted(glob.glob(f'{cfg.output_dir}/*')):
    print(f'  {os.path.basename(f):35s} {os.path.getsize(f)/1024:.1f} KB')